In [1]:
import os
from getpass import getpass
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
import random

from transformers import AutoTokenizer


In [2]:
# Check to see if files are downloaded before running the following cell
!ls -la data

total 179884
drwxr-xr-x 2 root root      4096 Jul 10 15:10 .
drwxr-xr-x 1 root root      4096 Jul 10 15:10 ..
-rw-r--r-- 1 root root       237 Oct  9  2024 sample_submission.csv
-rw-r--r-- 1 root root     10658 Oct  9  2024 test.csv
-rw-r--r-- 1 root root 184175071 Oct  9  2024 train.csv


In [3]:
# Get Kaggle API Token
os.environ['KAGGLE_API_TOKEN'] = getpass("Kaggle API token: ")

# Download kaggle competition datasets
!kaggle competitions download -c llm-classification-finetuning

# Unzip the files
!unzip -q llm-classification-finetuning.zip -d data

# Lists the files
!ls -la data

llm-classification-finetuning.zip: Skipping, found more recently modified local copy (use --force to force download)
replace data/sample_submission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: total 179884
drwxr-xr-x 2 root root      4096 Jul 10 15:10 .
drwxr-xr-x 1 root root      4096 Jul 10 15:10 ..
-rw-r--r-- 1 root root       237 Oct  9  2024 sample_submission.csv
-rw-r--r-- 1 root root     10658 Oct  9  2024 test.csv
-rw-r--r-- 1 root root 184175071 Oct  9  2024 train.csv


In [3]:

df = pd.read_csv("data/train.csv")
#df = df.head(200)

# Archetecure of model:

### input variables
```
prompt = prompt
response_1 = either reponse_a or response_b
response_2 = the other response_a/b
```
### Encoding model
```
model_1 = DeBERTA v3 base
```

### Classification model
```
model_2 = classification model
```
### pipeline of data
```
prompt -> model_1 -> output_p
response_1 -> model_1 -> output_1
response_2 -> model_1 -> output_2

(output_p, output_1, output_2) -> model_2 -> (prob of first response, prob of second response, prob of tie)
```



In [4]:
class LLMModelsPref(Dataset):
    def __init__(self, prompts, responses_a, responses_b, labels, tokenizer, max_len = 512, prob = 0.5):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.prob = prob
        self.prompts = prompts
        self.responses_a = responses_a
        self.responses_b = responses_b
        self.labels = labels



    def __len__(self):
        return len(self.prompts)
    
    def __getitem__(self, idx):
        prompt = self.prompts[idx]
        label = self.labels[idx]
        response_a = self.responses_a[idx]
        response_b = self.responses_b[idx]

        if random.random() < self.prob:
            response_a, response_b = response_b, response_a
            if label == 0:
                label = 1
            elif label ==1:
                label = 0


        encoded_p = self.tokenizer(prompt, max_length = self.max_len, truncation = True, padding = "max_length", return_tensors = "pt")
        encoded_a = self.tokenizer(response_a, max_length = self.max_len, truncation = True, padding = "max_length", return_tensors = "pt")
        encoded_b = self.tokenizer(response_b, max_length = self.max_len, truncation = True, padding = "max_length", return_tensors = "pt")

        prompt = encoded_p["input_ids"].squeeze(0)
        response_a = encoded_a["input_ids"].squeeze(0)
        response_b = encoded_b["input_ids"].squeeze(0)

        prompt_mask = encoded_p["attention_mask"].squeeze(0)
        response_a_mask = encoded_a["attention_mask"].squeeze(0)
        response_b_mask = encoded_b["attention_mask"].squeeze(0)

        label = torch.tensor(label, dtype = torch.long)

        return {
            "prompt": prompt, "prompt_mask": prompt_mask, 
            "response_a": response_a, "response_a_mask": response_a_mask, 
            "response_b": response_b, "response_b_mask": response_b_mask, 
            "label": label
        }
        



In [13]:
labels_one_hot = df[["winner_model_a","winner_model_b","winner_tie"]]
labels = labels_one_hot.values.argmax(axis = 1)

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

dataset = LLMModelsPref(
    prompts = df["prompt"].tolist(),
    responses_a = df["response_a"].tolist(),
    responses_b = df["response_b"].tolist(),
    labels= labels,
    tokenizer = tokenizer
)


Processing 57477 prompts...


: 

: 

: 

In [10]:
print(len(dataset))

dict = dataset[0]
print(dict["prompt_mask"])

200
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [11]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.1Gi       8.7Gi       2.0Mi       1.8Gi        10Gi
Swap:             0B          0B          0B
